# Stocks — Pipeline Playground

**This notebook is yours.** The scaffolding is built; the modelling is not.

It follows the same shape as your ENSF 444 final project notebook — Step 0
imports, data input, processing, `ColumnTransformer` → `Pipeline`,
`cross_validate` baseline, `GridSearchCV`, then results — so the structure
should be familiar.

**The one thing that is different:** every split here respects time. Your
notebooks used `train_test_split(random_state=0)` and `StratifiedKFold`, which
shuffle. That is correct for independent rows and catastrophic for dated,
overlapping ones. See `docs/project-charter.md` §7 for why. Expect your scores
to look *worse* than the coursework's. They will be real.

**Before you tune anything, calibrate.** A 30-day equity return is close to
unpredictable:

| metric | chance | a real result | implausible |
|---|---|---|---|
| R² | 0 | 0.005 – 0.02 | > 0.15 |
| directional accuracy | 50% | 52 – 55% | > 60% |
| information coefficient | 0 | 0.03 – 0.05 | > 0.15 |

## Step 0: Import Libraries

In [ ]:
import warnings
warnings.filterwarnings('ignore')  # suppressing convergence/deprecation noise

import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))  # so `import stocks` works from notebooks/

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import cross_validate, GridSearchCV
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

from stocks.features.assemble import build_panel, META_PREFIX, TEXT_PREFIX
from stocks.ingest.registry import Registry
from stocks.pipeline import build_pipeline, baseline_models, param_grid, split_columns
from stocks.evaluate import PurgedTimeSeriesSplit, purge_gap_rows, evaluate_pipeline, regression_report

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 50)

## Part 1: Data Input

Load a panel. Two options:

- **Synthetic** — instant, offline, and price/text are independent by
  construction, so any real score means leakage. Start here.
- **Real** — build it first from the CLI, which is slow enough that you want it
  cached rather than rebuilt in a notebook:

```
py run.py build-panel --symbols AAPL,MSFT,NVDA,JPM,XOM --start 2021-01-01
```

In [ ]:
from datetime import date, timedelta

USE_SYNTHETIC = True   # flip to False once you have built a real panel

if USE_SYNTHETIC:
    end = date.today() - timedelta(days=60)
    start = end - timedelta(days=900)
    panel = build_panel(['DEMO1', 'DEMO2', 'DEMO3', 'DEMO4'], start, end,
                        step_days=14, registry=Registry(force_synthetic=True),
                        progress=False)
else:
    panel = pd.read_parquet('../data/processed/panel.parquet')

print(f'panel: {panel.shape[0]} rows x {panel.shape[1]} columns')
panel[[f'{META_PREFIX}symbol', f'{META_PREFIX}as_of', 'target']].head()

## Part 2: Data Processing

Same checks as the coursework — missing values, dtypes, ranges, target
distribution — but with one addition that matters here: **look at the target
before you look at anything else.**

In [ ]:
# The target: 21-trading-day forward log return.
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

panel['target'].hist(bins=50, ax=axes[0], color='steelblue', edgecolor='black')
axes[0].axvline(0, color='tomato', linestyle='--')
axes[0].set_title('Target distribution (30-day forward log return)')
axes[0].set_xlabel('log return')

panel.plot(x=f'{META_PREFIX}as_of', y='target', ax=axes[1], legend=False,
           color='steelblue', alpha=0.7)
axes[1].axhline(0, color='tomato', linestyle='--')
axes[1].set_title('Target over time')

plt.tight_layout()
plt.show()

print(panel['target'].describe())
print(f"\nshare positive: {(panel['target'] > 0).mean():.1%}")

Note the target is roughly centred on zero and roughly symmetric. That is the
problem in one picture: there is no class imbalance to exploit and no obvious
structure to latch onto. A model that predicts ~0 everywhere will have a
respectable RMSE and zero value, which is why `#ML-3` (the zero baseline)
comes before everything else.

In [ ]:
# Missing values, by feature group. Structural NaN is expected — a window with
# no news genuinely has no sentiment — but a column that is ALWAYS NaN is a bug
# in the feature code, not missing data.
numeric_cols, text_cols = split_columns(panel)
print(f'{len(numeric_cols)} numeric features, {len(text_cols)} text blocks')

missing = panel[numeric_cols].isna().mean().sort_values(ascending=False)
print('\nmost-missing features:')
print(missing.head(10))

always_nan = missing[missing == 1.0]
if len(always_nan):
    print(f'\n!! {len(always_nan)} columns are entirely NaN — investigate:')
    print(list(always_nan.index))

In [ ]:
# Feature groups, by prefix. The pipeline routes on these.
groups = {}
for c in numeric_cols:
    groups.setdefault(c.split('_')[0], []).append(c)

for prefix, cols in sorted(groups.items(), key=lambda kv: -len(kv[1])):
    print(f'  {prefix:<10} {len(cols):>4} features')

## Part 3: Preprocessing — ColumnTransformer & Pipeline

Already built, in `stocks/pipeline.py`. The numeric branch is
median-impute → StandardScaler; each text block gets its own
TfidfVectorizer → AdaptiveSVD.

Median rather than mean (your Group20 notebook used mean): financial features
have heavy tails, and a mean imputed into a fat-tailed column lands at a value
no real observation ever took.

In [ ]:
pipe = build_pipeline(panel)
pipe

## Part 4: Model Training and Evaluation

### Step 4.1: Cross-Validation Baseline Comparison

Same idea as the coursework's first pass, with `PurgedTimeSeriesSplit` in place
of `StratifiedKFold`.

In [ ]:
X = panel.drop(columns=['target'])
y = panel['target']

gap = purge_gap_rows(step_days=14, horizon_days=21)  # match your build_panel step
cv = PurgedTimeSeriesSplit(n_splits=5, gap=gap)
print(f'purge gap: {gap} rows')

results_list = []
for name, model in baseline_models().items():
    scores = cross_validate(build_pipeline(panel, model), X, y, cv=cv,
                            scoring='r2', return_train_score=True, n_jobs=-1)
    results_list.append({
        'Model': name,
        'Training R2': round(scores['train_score'].mean(), 4),
        'Validation R2': round(scores['test_score'].mean(), 4),
    })

results = pd.DataFrame(results_list).set_index('Model')
print(results)

A large gap between training and validation R² is overfitting — and with ~150
collinear features it is the default outcome, not an edge case. Watch the
Random Forest in particular.

### Step 4.2: The comparison that actually matters (#ML-3)

**Do this before any tuning.** If none of the models beat *predict zero*, the
pipeline is not adding information and hyperparameter search will not change
that.

In [ ]:
# TODO #ML-3: implement the baselines and compare properly.
from sklearn.dummy import DummyRegressor

zero_baseline = DummyRegressor(strategy='constant', constant=0.0)
mean_baseline = DummyRegressor(strategy='mean')

for name, m in [('predict zero', zero_baseline), ('predict mean', mean_baseline)]:
    r = evaluate_pipeline(build_pipeline(panel, m), panel, n_splits=5, step_days=14)
    print(f"{name:<14} R2 {r.pooled['r2']:>8.4f}   IC {r.pooled['information_coefficient']:>7.4f}")

# Now compare against your best model. Does it beat these? If not, stop and
# think about features rather than about hyperparameters.

### Step 4.3: Grid Search

Your Group20 notebook's multi-model `param_grid` pattern, scored on purged
time-series folds.

`scoring='r2'` is the default here, but consider a custom scorer built on
`information_coefficient` — it is rank-based and therefore not dominated by the
handful of large moves that dominate squared error.

In [ ]:
grid = GridSearchCV(
    estimator=build_pipeline(panel),
    param_grid=param_grid(),
    cv=cv,
    scoring='r2',
    return_train_score=True,
    n_jobs=-1,
)
grid.fit(X, y)

print('Best parameters:', grid.best_params_)
print(f"Best CV train R2: {grid.cv_results_['mean_train_score'][grid.best_index_]:.4f}")
print(f"Best CV valid R2: {grid.best_score_:.4f}")

### Step 4.4: Full walk-forward report

`evaluate_pipeline` gives per-fold and pooled metrics, and prints the sanity
commentary alongside the numbers so a run is hard to over-read.

In [ ]:
result = evaluate_pipeline(build_pipeline(panel, grid.best_estimator_.named_steps['model']),
                           panel, n_splits=5, step_days=14)
print(result.summary())

## Part 5: Visualize Results

### Step 5.1: Predicted vs. actual

In [ ]:
best = build_pipeline(panel, grid.best_estimator_.named_steps['model'])

preds, actuals = [], []
for train_idx, valid_idx in cv.split(X):
    best.fit(X.iloc[train_idx], y.iloc[train_idx])
    preds.append(best.predict(X.iloc[valid_idx]))
    actuals.append(y.iloc[valid_idx].to_numpy())

pred = np.concatenate(preds)
actual = np.concatenate(actuals)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(pred, actual, alpha=0.4, color='steelblue', edgecolor='none')
axes[0].axhline(0, color='grey', lw=0.5); axes[0].axvline(0, color='grey', lw=0.5)
lims = [min(pred.min(), actual.min()), max(pred.max(), actual.max())]
axes[0].plot(lims, lims, 'r--', lw=1, label='perfect')
axes[0].set_xlabel('predicted'); axes[0].set_ylabel('actual')
axes[0].set_title('Out-of-fold predictions vs. actual')
axes[0].legend()

# The shape to look for: predictions clustered in a narrow band around zero
# while actuals spread wide. That is a model correctly reporting that it does
# not know, and it is the honest outcome for this problem.
axes[1].hist([pred, actual], bins=30, label=['predicted', 'actual'],
             color=['steelblue', 'tomato'])
axes[1].set_title('Prediction vs. actual spread')
axes[1].legend()

plt.tight_layout(); plt.show()

print(pd.Series(regression_report(actual, pred)).round(4))

### Step 5.2: Which feature groups carry the signal?

In [ ]:
model = grid.best_estimator_.named_steps['model']
prep = grid.best_estimator_.named_steps['preprocessor']

if hasattr(model, 'feature_importances_'):
    names = prep.get_feature_names_out()
    imp = pd.DataFrame({'Feature': names, 'Importance': model.feature_importances_})

    # By group, which is the more informative view: whether DS2/DS3 text matters
    # at all is a bigger question than which individual column ranks 7th.
    imp['Group'] = imp['Feature'].str.extract(r'__(\w+?)_')[0].fillna('other')
    by_group = imp.groupby('Group')['Importance'].sum().sort_values(ascending=False)

    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    by_group.plot(kind='barh', ax=axes[0], color='steelblue')
    axes[0].set_title('Total importance by feature group')

    top = imp.nlargest(20, 'Importance')
    sns.barplot(data=top, x='Importance', y='Feature', ax=axes[1], palette='Blues_r')
    axes[1].set_title('Top 20 individual features')

    plt.tight_layout(); plt.show()
else:
    coefs = pd.Series(model.coef_, index=prep.get_feature_names_out())
    print(coefs.abs().nlargest(20))

## Part 6: The ablation (#ML-6)

**The most informative experiment in this project.** Does the text actually add
anything over the price features alone?

If numeric+text does not beat numeric-only, DS2/DS3/DS5 are not earning their
keep. That is a real finding — report it rather than burying it.

In [ ]:
# TODO #ML-6
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from stocks.pipeline import build_preprocessor

numeric_cols, text_cols = split_columns(panel)

configurations = {
    'numeric only': (numeric_cols, []),
    'text only':    ([], text_cols),
    'both':         (numeric_cols, text_cols),
}

for label, (nc, tc) in configurations.items():
    pipe_ab = Pipeline([
        ('preprocessor', build_preprocessor(nc, tc)),
        ('model', Ridge(alpha=10.0, random_state=0)),
    ])
    r = evaluate_pipeline(pipe_ab, panel, n_splits=5, step_days=14)
    print(f"{label:<14} R2 {r.pooled['r2']:>8.4f}   "
          f"IC {r.pooled['information_coefficient']:>7.4f}   "
          f"dir {r.pooled['directional_accuracy']:>6.1%}")

## Part 7: Where to go next

Work through `docs/TODO.md`. In rough order of expected value:

1. **#ML-3** the zero baseline — the comparison everything else is measured against
2. **#ML-6** the ablation above — does the text matter?
3. **#ML-8** predict the Chapter 8 *residual* instead of the raw return. A model
   predicting raw returns spends most of its capacity re-learning "the market
   moved", which DS4 already tells you. This reframes the question as the one
   the news datasets can actually answer.
4. **#ML-10** quantile regression, so `/api/predict` can return an honest fan
   chart instead of a falsely confident point estimate
5. **#ML-11** a sequence model over raw daily bars — the interesting version,
   worth doing once there is a tabular number to beat

### Saving a model for the API

`/api/predict` loads `models/forecaster.joblib` if it exists and falls back to a
shrunk trailing-drift baseline otherwise, flagging itself `trained: false`.

In [ ]:
# import joblib
# from pathlib import Path
# Path('../models').mkdir(exist_ok=True)
# best.fit(X, y)   # refit on everything before saving
# joblib.dump(best, '../models/forecaster.joblib')
# print('saved — /api/predict will pick it up on the next request')